In [2]:

import numpy as np
import networkx as nx
from itertools import combinations
from time import time
import pickle
from argparse import Namespace
from scipy.optimize import minimize, OptimizeResult

from qiskit import QuantumCircuit, generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.converters import dag_to_circuit, circuit_to_dag
from qiskit.circuit import Parameter
from qiskit.transpiler import Layout

from qiskit_ibm_runtime import Session, SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeBrisbane 
 
from qopt_best_practices.transpilation.qaoa_construction_pass import QAOAConstructionPass

# from qiskit_qaoa.utils.qaoa_circuit_utils import get_mixer_operator, state_prep
from qiskit_qaoa.utils.transpiler_passes import ExtendedSwapStrategy
from qiskit_qaoa.utils.pass_managers import get_hubo_pass_manager
from qiskit_qaoa.utils.string_utils import evaluate_sparse_pauli_samples
from qiskit_qaoa.utils.logging import get_logger


def print_circuit_info(qc, circuit_name):
    logger.info(f'{circuit_name} has {qc.count_ops().get("cz", 0) + qc.count_ops().get("rzz", 0) + qc.count_ops().get("cx", 0)} 2Q gates \
    and {qc.depth(lambda instr: len(instr.qubits) > 1)} 2Q depth')
        

logger = get_logger(__name__)



args = Namespace(**dict(
    filename='trivial',
    extra=1,
    fraction_four=0.0,
    fraction_six=1.0,
    timeout=5,
    copy_numbers=[1,1,1],
    times_to_keep=(0,1),
    reps=4,
    method='COBYLA',
    shots=10000,
    init='random',
    nodes=3,
    time=3,
    alpha=0.05,
    swap_depth=40
))

logger.info(args)

filename: str = args.filename
p: int = args.reps
shots: int = args.shots
init_type: str = args.init
swap_depth: int = args.swap_depth
N: int = args.nodes
T: int = args.time
n = int(np.ceil(np.log2(2*N+1)))

seed = 1
rng = np.random.default_rng()

basis_gates=["sx", "x", "rz", "rzz", "cz", "id", "swap", "cx", "h"]


basepath = '/lustre/scratch127/qpg/jc59/hubo_hardware/'
filename = 'compilation.{}.extra{}.times{}.four{}.six{}'.format(
    args.filename,
    args.extra,
    ''.join([str(t) for t in args.times_to_keep]),
    args.fraction_four,
    args.fraction_six
)
results_file = basepath + filename + '.pkl'

with open(results_file, 'rb') as f:
    data = pickle.load(f)
    compiled_hamiltonian: SparsePauliOp = data['compiled_hamiltonian']
    full_hamiltonian: SparsePauliOp = data['full_hamiltonian']
    layout: Layout = data[swap_depth]

num_qubits: int = full_hamiltonian.num_qubits if full_hamiltonian.num_qubits is not None else max(layout.get_physical_bits().keys())

rows, cols = 0, 0
while 2 * (rows + cols + rows * cols) < num_qubits:
    if rows < cols:
        rows += 1
    else:
        cols += 1
logger.info(f'Min size to support virtual qubits: {(rows, cols)}, ')

extended_swap_strat = ExtendedSwapStrategy.from_heavy_hex(rows, cols)
num_physical_qubits = extended_swap_strat._num_vertices
coupling_map = extended_swap_strat._coupling_map

backend = FakeBrisbane()
sampler = Sampler(mode=backend)
logger.info(f'Backend: {backend}')
logger.info(f'Num qubits in backend: {backend.configuration().to_dict()["n_qubits"]}')

donor_qc = QuantumCircuit(num_physical_qubits)
remapped_full_hamiltonian = full_hamiltonian.apply_layout([layout.get_virtual_bits()[donor_qc.qubits[i]] for i in range(num_qubits)], num_physical_qubits)


logger.info(f'Physical qubits: {num_physical_qubits}')

coupling_map_edge = list(coupling_map)
physical_qubits = list(coupling_map.physical_qubits)
dual_coupling_map = nx.Graph()

for qubit in physical_qubits:
    edges = [edge for edge in coupling_map_edge if edge[0]==qubit]
    for edge1, edge2 in combinations(edges, 2):
        dual_coupling_map.add_edge(tuple(sorted(edge1)), tuple(sorted(edge2)))
edge_colouring = nx.greedy_color(dual_coupling_map, interchange=True)


pm = get_hubo_pass_manager(extended_swap_strat, swap_depth, args.extra)

cost_qc = QuantumCircuit(num_physical_qubits)
cost_qc.append(PauliEvolutionGate(compiled_hamiltonian, time=Parameter("c")), [layout.get_virtual_bits()[donor_qc.qubits[i]] for i in range(num_qubits)])
tcost_qc = pm.run(cost_qc)

print_circuit_info(tcost_qc, 'Transpiled cost hamiltonian circuit')
print(tcost_qc.count_ops())
logger.info(f'Cost hamiltonian circuit has {tcost_qc.num_qubits} qubits')


# TODO: instead of using construction pass, use p different cost hamiltonians with different mappings
# Can't do different mappings since the qubit locations are now set.. but could do the next N layers of SWAP strat
# Which would allow for a different subset of interactions to be used

if not 2*N+1 == 2**(int(np.log2(2*N+1))):
    # sp = state_prep(N,T)
    # mixer = get_mixer_operator(N,T)
    # logger.info('Using Grover mixer and state prep')
    sp = None
    mixer = None
    logger.info('Using X mixer and Hadamard state prep')
else:
    sp = None
    mixer = None
    logger.info('Using X mixer and Hadamard state prep')
    
    
construction_pass = QAOAConstructionPass(p, init_state=sp, mixer_layer=mixer)
qaoa_circ = dag_to_circuit(construction_pass.run(circuit_to_dag(tcost_qc)))

# Now transpile to basis gates
generic_pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
init  = generic_pm.init
init.remove(3)
generic_pm.init = init
# generic_pm.layout = None
t_qaoa_circ = generic_pm.run(qaoa_circ)

print_circuit_info(t_qaoa_circ, 'QAOA circuit')
logger.info(t_qaoa_circ.count_ops())
logger.info(f'QAOA circuit has {t_qaoa_circ.num_qubits} qubits')


qaoa_depth = len(t_qaoa_circ.parameters) // 2
if init_type == 'ramp':
    t = 0.7 * p
    betas = np.linspace(
        (1 / p) * (t * (1 - 0.5 / p)), (1 / p) * (t * 0.5 / p), p
    )
    gammas = betas[::-1]
    init_params = betas.tolist() + gammas.tolist()
elif init_type == 'warm':
    raise Exception('Warm start not implemented')
else:
    init_params = rng.uniform(0, 1, qaoa_depth).tolist() + rng.uniform(0, 1, qaoa_depth).tolist()
logger.info(f'Init: {init_params}')


logger.info(f'Noise model: {getattr(backend.options, "noise_model", "Ideal noise")}')

history = []

def cvar(energies, alpha=1.0):
    sorted_energies = sorted(energies)
    end_idx = int(alpha * len(energies))
    return np.sum(sorted_energies[0:end_idx]) / end_idx


def objective(x: np.ndarray):
    start = time()
    assigned_circuit = t_qaoa_circ.assign_parameters(x, inplace=False)
    sampler_job = sampler.run([assigned_circuit], shots=shots)
    sampler_result = sampler_job.result()
    counts = sampler_result[0].data.c.get_counts()
    sampling_time = time() - start
    start = time()
    energies = []
    evals = evaluate_sparse_pauli_samples(counts.keys(), remapped_full_hamiltonian)
    energies = [count * [evals[idx]] for idx, count in enumerate(counts.values())]
    flat_energies = [x for xs in energies for x in xs]
    total_energy = cvar(flat_energies, args.alpha)

    classical_post_process_time = time() - start
    history.append((sampling_time, total_energy, x.tolist(), counts, classical_post_process_time))
    return total_energy


11:26:23 - __main__ - INFO - Namespace(filename='trivial', extra=1, fraction_four=0.0, fraction_six=1.0, timeout=5, copy_numbers=[1, 1, 1], times_to_keep=(0, 1), reps=4, method='COBYLA', shots=10000, init='random', nodes=3, time=3, alpha=0.05, swap_depth=40)
11:26:23 - __main__ - INFO - Min size to support virtual qubits: (1, 2), 
11:26:23 - __main__ - INFO - Backend: <qiskit_ibm_runtime.fake_provider.backends.brisbane.fake_brisbane.FakeBrisbane object at 0x7f9feafdc190>
11:26:23 - __main__ - INFO - Num qubits in backend: 127


11:26:23 - __main__ - INFO - Physical qubits: 21
Max layers needed to apply swap decompose: 37
Gates we cannot directly implement: 7
[(7, 12, 13, 14), (2, 6, 7, 18), (3, 6, 7, 14, 18), (3, 7, 18), (7, 13, 14, 19), (2, 3, 7, 18), (2, 7, 18)]
Transpiling accumulator
11:26:24 - __main__ - INFO - Transpiled cost hamiltonian circuit has 262 2Q gates     and 220 2Q depth


INFO:__main__:Transpiled cost hamiltonian circuit has 262 2Q gates     and 220 2Q depth


OrderedDict([('swap', 316), ('cx', 262), ('rz', 91)])
11:26:24 - __main__ - INFO - Cost hamiltonian circuit has 21 qubits


INFO:__main__:Cost hamiltonian circuit has 21 qubits


...

INFO:__main__:QAOA circuit has 127 qubits


11:26:41 - __main__ - INFO - Init: [0.41176500366671853, 0.6202823806965307, 0.3921660794166183, 0.5547920700044152, 0.42432341513758454, 0.4720510211537723, 0.43728219436069315, 0.0581926702136617]


INFO:__main__:Init: [0.41176500366671853, 0.6202823806965307, 0.3921660794166183, 0.5547920700044152, 0.42432341513758454, 0.4720510211537723, 0.43728219436069315, 0.0581926702136617]


11:26:41 - __main__ - INFO - Noise model: None


INFO:__main__:Noise model: None


In [ ]:
x0 = np.array([0.22521636, 0.29549166, 0.06373729, 0.39703442, 0.23780506, 0.41909703,
 0.22658979, 0.65521582])
objective(x0)